<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-02-grafos/13%20-%20Algoritmos%20de%20Busca%20BFS%20e%20DFS%20em%20Tubulacoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 13 - Notebook: Busca em Largura (BFS) e Busca em Profundidade (DFS) em Redes de Hidrogênio (SCADA H₂)

Neste notebook implementamos os algoritmos **BFS** e **DFS** para a **Estação de Reabastecimento de Hidrogênio**, permitindo a descoberta de rotas operacionais com menor quantidade de válvulas a comutar e a enumeração de caminhos de contingência sob falha de equipamentos.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII pura."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoTubulacao:
    def __init__(self, vertices):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n): self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes = []

    def adicionar_tubulacao(self, origem, destino, comprimento_m, tag_valvula, diametro_pol=1.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Origem": origem, "Destino": destino,
            "Comprimento (m)": comprimento_m, "Válvula ISA": tag_valvula, "Diâmetro (pol)": diametro_pol
        })

def criar_rede_estacao_h2():
    nos = ["E-101", "C-101", "C-102", "TK-101", "TK-102", "CH-101", "MAN-101", "D-101"]
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao("E-101", "C-101", 5.0, "XV-101", 2.0)
    g.adicionar_tubulacao("C-101", "TK-101", 10.0, "XV-102", 1.0)
    g.adicionar_tubulacao("C-101", "C-102", 15.0, "XV-103", 1.0)
    g.adicionar_tubulacao("C-102", "TK-102", 8.0, "XV-104", 0.5)
    g.adicionar_tubulacao("TK-101", "MAN-101", 12.0, "XV-105", 1.0)
    g.adicionar_tubulacao("TK-102", "MAN-101", 15.0, "XV-106", 0.5)
    g.adicionar_tubulacao("CH-101", "MAN-101", 6.0, "XV-107", 1.0)
    g.adicionar_tubulacao("MAN-101", "D-101", 4.0, "XV-108", 0.5)
    return g

from collections import deque
from typing import List, Set, Optional

rede = criar_rede_estacao_h2()

class NavegadorGrafos:
    def __init__(self, grafo: GrafoTubulacao):
        self.g = grafo

    def bfs_menor_numero_valvulas(self, origem: str, destino: str, 
                                  nos_bloqueados: Optional[Set[str]] = None) -> Optional[List[str]]:
        if nos_bloqueados is None: nos_bloqueados = set()
        if origem in nos_bloqueados or destino in nos_bloqueados: return None
        
        fila = deque([(origem, [origem])])
        visitados = {origem}
        while fila:
            u_nome, caminho = fila.popleft()
            if u_nome == destino: return caminho
            u_idx = self.g.v_to_idx[u_nome]
            for v_idx in range(self.g.n):
                v_nome = self.g.idx_to_v[v_idx]
                if self.g.adj_binaria[u_idx][v_idx] == 1 and v_nome not in visitados and v_nome not in nos_bloqueados:
                    visitados.add(v_nome)
                    fila.append((v_nome, caminho + [v_nome]))
        return None

    def dfs_todos_os_caminhos(self, origem: str, destino: str, 
                             nos_bloqueados: Optional[Set[str]] = None) -> List[List[str]]:
        if nos_bloqueados is None: nos_bloqueados = set()
        caminhos = []
        def dfs(u_nome: str, visitados: List[str]):
            if u_nome == destino:
                caminhos.append(list(visitados))
                return
            u_idx = self.g.v_to_idx[u_nome]
            for v_idx in range(self.g.n):
                v_nome = self.g.idx_to_v[v_idx]
                if self.g.adj_binaria[u_idx][v_idx] == 1 and v_nome not in visitados and v_nome not in nos_bloqueados:
                    visitados.append(v_nome)
                    dfs(v_nome, visitados)
                    visitados.pop()
        dfs(origem, [origem])
        return caminhos

nav = NavegadorGrafos(rede)
rota_bfs = nav.bfs_menor_numero_valvulas("E-101", "D-101")
print("1. Rota com menor número de válvulas (BFS):", " -> ".join(rota_bfs))

todas_rotas = nav.dfs_todos_os_caminhos("E-101", "D-101")
print(f"\n2. Total de opções de rotas (DFS): {len(todas_rotas)}")
for i, r in enumerate(todas_rotas, 1):
    print(f"  Opção {i}: {' -> '.join(r)}")

rota_cont = nav.bfs_menor_numero_valvulas("E-101", "D-101", nos_bloqueados={"TK-101"})
print("\n3. Rota de contingência (Bloqueio do Banco LP TK-101):", " -> ".join(rota_cont))
assert "TK-102" in rota_cont

rota_falha = nav.bfs_menor_numero_valvulas("E-101", "D-101", nos_bloqueados={"C-101"})
if rota_falha is None:
    print("\n4. Simulação de falha crítica (Bloqueio do Compressor C-101): Sem rota disponível (Falha no ponto único de produção)")


1. Rota com menor número de válvulas (BFS): E-101 -> C-101 -> TK-101 -> MAN-101 -> D-101

2. Total de opções de rotas (DFS): 2
  Opção 1: E-101 -> C-101 -> TK-101 -> MAN-101 -> D-101
  Opção 2: E-101 -> C-101 -> C-102 -> TK-102 -> MAN-101 -> D-101

3. Rota de contingência (Bloqueio do Banco LP TK-101): E-101 -> C-101 -> C-102 -> TK-102 -> MAN-101 -> D-101

4. Simulação de falha crítica (Bloqueio do Compressor C-101): Sem rota disponível (Falha no ponto único de produção)
